# Bull Re-entry Participation Target Gate

Purpose: add a fold-local training target for bull re-entry participation without relaxing the final walk-forward retention gate.

In [1]:

from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'SPEC.md').exists():
            return candidate
    raise FileNotFoundError('could not locate repository root from notebook cwd')


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_training_targets import (
    BullReentryParticipationTargetConfig,
    apply_bull_reentry_participation_target_gate,
)

CARD = ROOT / 'runtime/model_selection/card/bull_reentry_participation_target_gate_v1'
CARD.mkdir(parents=True, exist_ok=True)

fixture = pd.DataFrame(
    {
        'datetime': pd.to_datetime([
            '2020-03-24',
            '2020-04-01',
            '2022-01-03',
            '2020-04-03',
            '2020-04-06',
        ]),
        'fold': ['wf_2020_train'] * 5,
        'window': ['wf_2020_train'] * 5,
        'target_scope': ['fold_local', 'fold_local', 'fold_local', 'fold_local', 'global'],
        'train_start': ['2020-01-01'] * 5,
        'train_end': ['2020-12-31'] * 5,
        'eval_start': ['2022-01-01'] * 5,
        'eval_end': ['2022-12-31'] * 5,
        'action': ['BUY', 'HOLD', 'BUY', 'BUY', 'BUY'],
        'teacher_position': [0.20, 0.35, 0.25, 0.50, 0.20],
        'rebound_target_position': [0.20, 0.35, 0.25, 0.50, 0.20],
        'risk_target_position_original': [0.20, 0.35, 0.25, 0.50, 0.20],
        'risk_target_position': [0.20, 0.35, 0.25, 0.50, 0.20],
        'objective_weight': [1.0, 2.0, 1.0, 5.0, 1.0],
        'rebound_participation_weight': [0.0, 0.0, 0.0, 1.0, 0.0],
        'is_rebound_cluster': [False, False, False, True, False],
        'is_downtrend_state': [False, False, False, True, False],
        'drawdown_control_state': [False, False, False, True, False],
        'release_gate_state': [False, True, False, True, False],
    }
)
config = BullReentryParticipationTargetConfig(
    enabled=True,
    min_target_position=0.70,
    max_target_position=0.85,
    objective_weight_bonus=5.0,
    include_release_gate=True,
    exclude_downtrend=True,
    exclude_drawdown_control=True,
)
result = apply_bull_reentry_participation_target_gate(fixture, config)
summary = {
    'experiment': 'bull_reentry_participation_target_gate_v1',
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'bull_reentry_target_gate_ready_for_fold_local_training_only',
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_next': True,
    'next_mechanism': 'fold_local_bull_reentry_bc_warm_start_then_full_walkforward_retention_gate',
    'required_retention_gates': {
        'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
        'max_walkforward_drawdown_pct': 20.0,
        'required_best_return_pct': 213.143462,
        'reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
        'reference_role': 'qlib_reference_baseline_not_actor',
    },
    'config': {
        'min_target_position': config.min_target_position,
        'max_target_position': config.max_target_position,
        'objective_weight_bonus': config.objective_weight_bonus,
        'include_release_gate': config.include_release_gate,
        'exclude_downtrend': config.exclude_downtrend,
        'exclude_drawdown_control': config.exclude_drawdown_control,
    },
    'rows': int(len(result)),
    'target_state_rows': int(result['bull_reentry_target_state'].sum()),
    'eval_overlap_target_rows': int(
        (
            result['bull_reentry_target_state']
            & (result['datetime'] >= pd.Timestamp('2022-01-01'))
            & (result['datetime'] <= pd.Timestamp('2022-12-31'))
        ).sum()
    ),
    'global_scope_target_rows': int(
        (result['bull_reentry_target_state'] & result['target_scope'].astype(str).eq('global')).sum()
    ),
    'risk_conflict_target_rows': int(
        (
            result['bull_reentry_target_state']
            & (result['is_downtrend_state'] | result['drawdown_control_state'])
        ).sum()
    ),
    'max_bull_reentry_target_lift': float(result['bull_reentry_target_lift'].max()),
    'mean_target_position_when_active': float(
        result.loc[result['bull_reentry_target_state'], 'risk_target_position'].mean()
        if result['bull_reentry_target_state'].any()
        else 0.0
    ),
    'conclusion': [
        'The gate creates fold-local bull re-entry participation labels from BUY or release-gate states.',
        'It does not touch evaluation rows, global-scope labels, or rows already flagged as downtrend/drawdown-control risk.',
        'This is a target-generation contract only; no actor is promotable until full walk-forward retention gates pass.',
    ],
}
columns = [
    'datetime',
    'fold',
    'target_scope',
    'action',
    'release_gate_state',
    'is_downtrend_state',
    'drawdown_control_state',
    'risk_target_position_original',
    'risk_target_position',
    'rebound_target_position',
    'bull_reentry_target_state',
    'bull_reentry_target_lift',
    'objective_weight',
    'rebound_participation_weight',
    'bull_reentry_target_reason',
]
result[columns].to_csv(CARD / 'target_gate_audit.csv', index=False)
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps({
    'decision': summary['decision'],
    'target_state_rows': summary['target_state_rows'],
    'eval_overlap_target_rows': summary['eval_overlap_target_rows'],
    'global_scope_target_rows': summary['global_scope_target_rows'],
    'risk_conflict_target_rows': summary['risk_conflict_target_rows'],
    'next_mechanism': summary['next_mechanism'],
}, indent=2))


{
  "decision": "bull_reentry_target_gate_ready_for_fold_local_training_only",
  "target_state_rows": 2,
  "eval_overlap_target_rows": 0,
  "global_scope_target_rows": 0,
  "risk_conflict_target_rows": 0,
  "next_mechanism": "fold_local_bull_reentry_bc_warm_start_then_full_walkforward_retention_gate"
}


## Conclusion

The gate is only a target-generation contract. It can feed BC/checkpoint scoring, but retained actors still require full WF windows, MaxDD < 20%, and return alignment against the SOXLBacktest reference baseline.